In [1]:
import pickle

with open("/raid/data/smunoz/catnip/EviCYP/data_splits/drug_bmfm.pkl", "rb") as f:
    drugdict = pickle.load(f)

sample_key = next(iter(drugdict.keys()))
print(f"Key: {sample_key}, Type: {type(sample_key)}")

Key: [H][C@@]1(C2(C)C)[C@@]([C@@]3([N+]#[C-])[C@](C=C)(C)CC1)(O)C4=C(C(C)(C)C=C3)NC5=CC=CC2=C54, Type: <class 'str'>


In [8]:
import sys
import numpy.core

# Map NumPy 2.x pickle artifacts to NumPy 1.x internals
sys.modules['numpy._core'] = numpy.core
sys.modules['numpy._core.numeric'] = numpy.core.numeric

import sys
import numpy._core as _core
import numpy._core.numeric as _numeric

sys.modules['numpy.core'] = _core
sys.modules['numpy.core.numeric'] = _numeric
import pickle

# 1. Verify Drug Pickle
with open("/raid/data/smunoz/catnip/EviCYP/data_splits/drug_bmfm.pkl", "rb") as f:
    drugdict = pickle.load(f)

sample_smiles = next(iter(drugdict))
print("--- DRUG PICKLE CHECK ---")
print(f"Key sample: {sample_smiles}")
print(f"Key type:   {type(sample_smiles)}")  # Expected: <class 'str'>
print(f"Embedding shape: {drugdict[sample_smiles].shape}") # Expected: (1236,)

print("\n" + "="*30 + "\n")

# 2. Verify Target Pickle
with open("/raid/data/smunoz/catnip/EviCYP/data_splits/target_esmc.pkl", "rb") as f:
    targetdict = pickle.load(f)

sample_target_id = next(iter(targetdict))
print("--- TARGET PICKLE CHECK ---")
print(f"Key sample: {sample_target_id}")
print(f"Key type:   {type(sample_target_id)}") # Expected: <class 'int'>
print(f"Dict keys:  {list(targetdict[sample_target_id].keys())}") # Expected: ['sequence', 'length', 'features']
print(f"Features shape: {targetdict[sample_target_id]['features'].shape}") # Expected: (L, 1152)

--- DRUG PICKLE CHECK ---
Key sample: [H][C@@]1(C2(C)C)[C@@]([C@@]3([N+]#[C-])[C@](C=C)(C)CC1)(O)C4=C(C(C)(C)C=C3)NC5=CC=CC2=C54
Key type:   <class 'str'>
Embedding shape: (1236,)


--- TARGET PICKLE CHECK ---
Key sample: 1
Key type:   <class 'int'>
Dict keys:  ['sequence', 'length', 'features']
Features shape: (329, 1152)


In [9]:
import pickle

DRUG_PKL = '/raid/data/smunoz/catnip/EviCYP/data_splits/drug_bmfm.pkl'
with open(DRUG_PKL, 'rb') as f:
    drugdict = pickle.load(f)

print("Total keys in drugdict:", len(drugdict))
print("Sample keys:", list(drugdict.keys())[:10])

Total keys in drugdict: 118
Sample keys: ['[H][C@@]1(C2(C)C)[C@@]([C@@]3([N+]#[C-])[C@](C=C)(C)CC1)(O)C4=C(C(C)(C)C=C3)NC5=CC=CC2=C54', '[H][C@@]1(C2(C)C)[C@@]([C@@H]([N+]#[C-])[C@](C=C)(C)CC1)(O)C3=C(C(C)(C)C=C)NC4=CC=CC2=C43', 'OC([C@@H]1CCC[NH2+]1)(C2=CC=CC=C2)C3=CC=CC=C3', 'O[C@@H](C1C[C@@H]2CC[N@]1CC2C=C)C3=CC=NC4=CC=C(OC)C=C43', '[O-]N(C1=O)C2=C(C=CC=C2)O[C@@H]1O[C@H]3[C@H](O)[C@@H](O)[C@H](O)[C@@H](CO)O3', 'O=C(C(OC)=C1)C=C(C)C2=C1O[C@@]3(C)[C@@](C2)([H])CC/C(C)=C/CC(C)(C)/C=C/C3', '[H][C@@]12N(C)C3=CC=CC([C@](C=C(C)C)([H])N(CC4)[C@H]5[NH2+]CC6)=C3[C@]14[C@@]56C7=C(C=CC=C7)N2', 'C=C(C)[C@](CC[C@](C=C)(C)[C@@H]1[N+]#[C-])([H])[C@@]1([H])C2=CNC3=CC=CC=C32', 'O=C1C(OC(C)=O)C2=C(C)CCC3=C(OC=C3)C2=C1C', 'N[C@@H](CC1=CNC2=C1C=CC=C2)C(OC)=O']


In [10]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import torch
import logging
from pathlib import Path

# RDKit imports
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')

# Import project root for BMFM model
project_root = Path(__file__).parent.parent.resolve()
sys.path.insert(0, str(project_root))

from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy
from bmfm_sm.predictive.data_modules.graph_finetune_dataset import Graph2dFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.image_finetune_dataset import ImageFinetuneDataPipeline
from bmfm_sm.predictive.data_modules.text_finetune_dataset import TextFinetuneDataPipeline

# Add current folder to sys.path to find evidential_DL and train modules
sys.path.append(os.path.dirname(os.path.abspath(__file__)))

from evidential_DL import dirichlet_uncertainty
from train import CYPClassifier, pad_or_trim_with_mask

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ================= Configuration & Paths =================
DATADIR = '/raid/data/smunoz/catnip/EviCYP/data_splits'
SUBSTRATES_CSV = '/raid/data/smunoz/catnip/data/subsrates.csv'
MODEL_PATH = '/raid/data/smunoz/catnip/EviCYP/model/results/data_splits/drug_bmfm-target_esmc/0/model.pth'
DRUG_PKL = os.path.join(DATADIR, 'drug_bmfm.pkl')
TARGET_PKL = os.path.join(DATADIR, 'target_esmc.pkl')
NORM_PARAMS_PATH = os.path.join(DATADIR, 'normalization_parameters.csv')

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Define benchmark substrates and their ground truth reactive enzymes
SUBSTRATES_CONFIG = {
    "Sparteine": {
        "type": "external",
        "smiles": "C1CC2CN3C1C4CCCC3C24",
        "key_enzymes": ["142", "299", "307", "304", "303", "302", "308", "305", "300", "301"]
    },
    "6-Methyleneandrost-4-ene-3,17-dione": {
        "type": "external",
        "smiles": "O=C1CC=C2C(=C)C3CCC4(C)C(=O)CCC4C3CC2(C)C1",
        "key_enzymes": ["115", "60", "52", "91", "96", "192", "274", "57", "58", "119"]
    },
    "Matridine": {
        "type": "external",
        "smiles": "C1CC2CN3C1C4CCCC3C24", # Verify exact Matridine SMILES string
        "key_enzymes": ["142", "299", "307", "302", "308", "303", "304", "300", "305", "306"]
    },
    "Substrate_008": {
        "type": "internal",
        "id": 8,
        "key_enzymes": ["1", "2", "5", "7", "8", "10", "11", "12", "15", "16", "17", "18", "20", "21", "22", "26", "27", "28", "29", "30", "31", "32", "33", "34", "38", "50", "55", "73", "87", "110", "123"]
    },
    "Substrate_034": {
        "type": "internal",
        "id": 34,
        "key_enzymes": ["57", "59", "60", "61", "64", "65", "69", "70", "73", "76", "78", "79", "84", "141", "288"]
    }
}

# ================= RDKit Descriptor Setup =================
EXCLUDED_DESCRIPTORS = {'SMR_VSA8', 'SlogP_VSA9', 'fr_isocyan', 'fr_prisulfonamd', 'Ipc'}
rdkit_desc_list = [(name, func) for name, func in Descriptors._descList if name not in EXCLUDED_DESCRIPTORS]
descriptor_names = [name for name, _ in rdkit_desc_list]

def load_normalization_params(norm_params_path):
    if not os.path.exists(norm_params_path):
        return {}
    norm_params_df = pd.read_csv(norm_params_path)
    return {row['descriptor']: (row['mean'], row['std']) for _, row in norm_params_df.iterrows()}

def compute_rdkit_descriptors(mol, norm_params=None):
    vals = []
    for name, func in rdkit_desc_list:
        try:
            v = func(mol)
            if norm_params and name in norm_params:
                mean, std = norm_params[name]
                v_norm = 0.0 if std == 0 else ((float(v) - mean) / std if v is not None else np.nan)
                vals.append(v_norm)
            else:
                vals.append(float(v) if v is not None else np.nan)
        except Exception:
            vals.append(np.nan)
    return np.array(vals, dtype=np.float32)

def compute_morgan_fp(mol, radius=2, nBits=512):
    bv = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=np.uint8)
    AllChem.DataStructs.ConvertToNumpyArray(bv, arr)
    return arr.astype(np.float32)

def generate_bmfm_embedding(smiles, bmfm_model):
    joint_dict = {}
    joint_dict.update(Graph2dFinetuneDataPipeline.smiles_to_graph_format(smiles))
    joint_dict.update(TextFinetuneDataPipeline.smiles_to_text_format(smiles))
    joint_dict.update(ImageFinetuneDataPipeline.smiles_to_image_format(smiles))
    joint_dict = {k: v.to('cpu') if torch.is_tensor(v) else v for k, v in joint_dict.items()}
    with torch.no_grad():
        emb = bmfm_model.get_embeddings(joint_dict)
    return emb.cpu().detach().numpy().astype(np.float32).ravel()

def smiles_to_feature_vector(smiles, bmfm_model, norm_params):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")
    emb_np = generate_bmfm_embedding(smiles, bmfm_model)
    desc_vals = compute_rdkit_descriptors(mol, norm_params)
    fp = compute_morgan_fp(mol, radius=2, nBits=512)
    return np.concatenate([emb_np, desc_vals, fp]).astype(np.float32)

def get_smiles_from_csv(csv_path, sub_id):
    df = pd.read_csv(csv_path)
    id_col = [c for c in df.columns if 'id' in c.lower()][0]
    smiles_col = [c for c in df.columns if 'smiles' in c.lower()][0]
    
    match = df[df[id_col].astype(str).str.lstrip('0') == str(sub_id).lstrip('0')]
    if not match.empty:
        return match.iloc[0][smiles_col]
    return None

def load_cyp_model(model_path, drug_dim=1236, target_dim=1152):
    model = CYPClassifier(drug_embed_dim=drug_dim, target_embed_dim=target_dim, drug_codebook_size=128, target_codebook_size=128, code_dim=4, n_class=2).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

def evaluate_all_enzymes(model, drug_vector, targetdict, target_max_len=520):
    drug_tensor = torch.tensor(drug_vector, dtype=torch.float32).unsqueeze(0).to(device)
    results = []

    with torch.no_grad():
        for target_id, target_data in targetdict.items():
            feat = target_data["features"]
            feat_pad, mask = pad_or_trim_with_mask(feat, target_max_len)
            target_tensor = feat_pad.unsqueeze(0).to(device)
            mask_tensor = mask.unsqueeze(0).to(device)

            log_probs, _, _, alpha, _, _ = model.predict(drug_tensor, target_tensor, mask_tensor)
            prob = torch.exp(log_probs)[0, 1].item()
            _, uncertainty = dirichlet_uncertainty(alpha)

            results.append({
                "Enzyme_ID": str(target_id),
                "PredProb": prob,
                "Uncertainty": uncertainty[0].item()
            })

    df = pd.DataFrame(results).sort_values(by="PredProb", ascending=False).reset_index(drop=True)
    df["Rank"] = df.index + 1
    return df

# ================= Main Execution =================
if __name__ == '__main__':
    logger.info("Loading target dictionary and drug embeddings...")
    with open(TARGET_PKL, 'rb') as f:
        targetdict = pickle.load(f)
    with open(DRUG_PKL, 'rb') as f:
        drugdict = pickle.load(f)

    cyp_model = load_cyp_model(MODEL_PATH)
    norm_params = load_normalization_params(NORM_PARAMS_PATH)

    logger.info("Loading BMFM model for SMILES embeddings...")
    bmfm_model = SmallMoleculeMultiViewModel.from_pretrained(
        LateFusionStrategy.ATTENTIONAL,
        model_path='ibm-research/biomed.sm.mv-te-84m',
        huggingface=True
    ).to('cpu')

    # Standardize true reactive enzyme IDs into string format
    for sub_name, data in SUBSTRATES_CONFIG.items():
        data["key_enzymes_str"] = set(str(e).lstrip('0') for e in data["key_enzymes"])

    for sub_name, data in SUBSTRATES_CONFIG.items():
        logger.info(f"Processing {sub_name}...")
        
        if data["type"] == "external":
            smiles = data["smiles"]
        else:
            smiles = get_smiles_from_csv(SUBSTRATES_CSV, data["id"])
            if not smiles:
                logger.warning(f"Could not find SMILES in CSV for substrate ID {data['id']}")
                continue

        if smiles in drugdict:
            drug_vector = drugdict[smiles]
        else:
            drug_vector = smiles_to_feature_vector(smiles, bmfm_model, norm_params)

        # Get full ranked list across all enzymes
        df_rankings = evaluate_all_enzymes(cyp_model, drug_vector, targetdict)

        # Select top 10 ranked predictions
        top10_df = df_rankings.head(10).copy()

        # Add Ground Truth (1 if in known reactive set, 0 otherwise)
        top10_df["Ground_Truth"] = top10_df["Enzyme_ID"].apply(
            lambda x: 1 if str(x).lstrip('0') in data["key_enzymes_str"] else 0
        )
        
        # Format enzyme name and values
        top10_df["Enzyme"] = top10_df["Enzyme_ID"].apply(lambda x: f"NHI_{int(x):03d}")
        top10_df["PredProb"] = top10_df["PredProb"].apply(lambda x: f"{x:.4f}")
        top10_df["Uncertainty"] = top10_df["Uncertainty"].apply(lambda x: f"{x:.4f}")

        # Display formatted output block
        print("\n" + "=" * 60)
        print(f" SUBSTRATE: {sub_name} (Top 10 Predictions)")
        print("=" * 60)
        print(top10_df[["Rank", "Enzyme", "Ground_Truth", "PredProb", "Uncertainty"]].to_string(index=False))

NameError: name '__file__' is not defined